In [27]:
import polars as pl
import numpy as np
import pandas as pd
import os
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

In [38]:
# ============================================================
# КОНФИГУРАЦИЯ (ОПТИМИЗИРОВАННАЯ ДЛЯ LIGHTGBM)
# ============================================================
SEED = 42
LOAD_BATCH_SIZE = 50_000
SUBSAMPLE_SIZE = 120_000
TOP_K_FEATURES = 300
N_FOLDS = 3                    # 3 фолда для скорости
SELECTED_FEATURES_PATH = "data/selected_features"
os.makedirs(SELECTED_FEATURES_PATH, exist_ok=True)

# Параметры LightGBM (оптимизированы для скорости и качества)
LGB_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'num_threads': 8,
    'min_data_in_leaf': 20,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'seed': SEED
}

# Параметры для сложных классов (больше мощности)
LGB_PARAMS_DEEP = {
    **LGB_PARAMS,
    'num_leaves': 64,
    'learning_rate': 0.04,
    'min_data_in_leaf': 10,
    'lambda_l1': 0.05,
    'lambda_l2': 0.05
}

# Включение GPU (если есть)
# Для GPU раскомментируйте:
# LGB_PARAMS['device'] = 'gpu'
# LGB_PARAMS['gpu_platform_id'] = 0
# LGB_PARAMS['gpu_device_id'] = 0


In [39]:
# Параметры для быстрого отбора (с GPU)
SELECT_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'verbose': -1,
    'num_threads': 8,
    'seed': SEED,
    'device': 'gpu',           # Включаем GPU
    'gpu_platform_id': 0,      # ID платформы (обычно 0)
    'gpu_device_id': 0         # ID устройства (0 = первая GPU)
}

# Основные параметры LightGBM (с GPU)
LGB_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'num_threads': 8,
    'min_data_in_leaf': 20,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'seed': SEED,
    'device': 'gpu',           # Включаем GPU
    'gpu_platform_id': 0,
    'gpu_device_id': 0
}

# Параметры для сложных классов (с GPU)
LGB_PARAMS_DEEP = {
    **LGB_PARAMS,
    'num_leaves': 64,
    'learning_rate': 0.04,
    'min_data_in_leaf': 10,
    'lambda_l1': 0.05,
    'lambda_l2': 0.05
}

In [ ]:
# ============================================================
# 1. ЗАГРУЗКА И ОБЪЕДИНЕНИЕ ДАННЫХ
# ============================================================
def build_wide_data():
    """Загружает и объединяет main и extra признаки"""
    print("Загрузка и объединение признаков...")
    
    train_customer_ids = pl.read_parquet('data/train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet('data/test_main_features.parquet').select('customer_id')
    
    lf_main_train = pl.scan_parquet('data/train_main_features.parquet').drop('customer_id')
    lf_extra_train = pl.scan_parquet('data/train_extra_features.parquet').drop('customer_id')
    lf_main_test = pl.scan_parquet('data/test_main_features.parquet').drop('customer_id')
    lf_extra_test = pl.scan_parquet('data/test_extra_features.parquet').drop('customer_id')
    
    train_wide = pl.concat([lf_main_train, lf_extra_train], how='horizontal').with_columns(
        pl.all().cast(pl.Float16)
    ).collect()
    
    test_wide = pl.concat([lf_main_test, lf_extra_test], how='horizontal').with_columns(
        pl.all().cast(pl.Float16)
    ).collect()
    
    print(f"Train wide shape: {train_wide.shape}")
    print(f"Test wide shape: {test_wide.shape}")
    return train_wide, test_wide, train_customer_ids, test_customer_ids

# Загружаем или создаем wide features
if not os.path.exists('data/train_wide_features.parquet'):
    print("Создаем train_wide_features.parquet...")
    train_wide, test_wide, train_customer_ids, test_customer_ids = build_wide_data()
    train_wide.write_parquet('data/train_wide_features.parquet')
    test_wide.write_parquet('data/test_wide_features.parquet')
    print("Файлы сохранены")
else:
    print("Загружаем существующие wide features...")
    train_wide = pl.read_parquet('data/train_wide_features.parquet')
    test_wide = pl.read_parquet('data/test_wide_features.parquet')
    train_customer_ids = pl.read_parquet('data/train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet('data/test_main_features.parquet').select('customer_id')

# Удаляем дубликаты столбцов
def remove_duplicate_columns(df):
    unique_cols = []
    seen = set()
    for col in df.columns:
        if col not in seen:
            unique_cols.append(col)
            seen.add(col)
    return df.select(unique_cols)

train_wide = remove_duplicate_columns(train_wide)
test_wide = remove_duplicate_columns(test_wide)
print(f"После удаления дубликатов: train shape {train_wide.shape}, test shape {test_wide.shape}")

Загружаем существующие wide features...
После удаления дубликатов: train shape (750000, 2440), test shape (250000, 2440)


In [41]:
# ============================================================
# 2. ФУНКЦИИ ДЛЯ ОТБОРА ПРИЗНАКОВ (LightGBM)
# ============================================================
def load_parquet_subsample(path: str, idx: np.ndarray) -> pl.DataFrame:
    """Загружает подвыборку строк из parquet файла"""
    parts = []
    offset = 0
    
    for batch in pq.ParquetFile(path).iter_batches(batch_size=LOAD_BATCH_SIZE):
        left = np.searchsorted(idx, offset)
        right = np.searchsorted(idx, offset + batch.num_rows)
        if left < right:
            df_part = pl.from_arrow(batch)[(idx[left:right] - offset).tolist()]
            parts.append(df_part)
        offset += batch.num_rows
    
    result = pl.concat(parts, how="vertical")
    cat_features = [col for col in result.columns if col.startswith("cat_feature")]
    for col in cat_features:
        result = result.with_columns(pl.col(col).cast(pl.Int32).fill_null(-1))
    
    return result

def subsample(target_size: int, y: np.ndarray) -> np.ndarray:
    """Создаёт сбалансированную подвыборку"""
    rng = np.random.default_rng(SEED)
    all_idx = np.arange(y.size)
    pos_idx = np.flatnonzero(y == 1)
    
    if pos_idx.size < target_size / 2:
        neg_idx = np.flatnonzero(y == 0)
        neg_take = rng.choice(neg_idx, size=target_size - pos_idx.size, replace=False)
        idx = np.concatenate((pos_idx, neg_take))
    else:
        idx = rng.choice(all_idx, size=target_size, replace=False)
    
    return np.sort(idx)

def select_topk_features_lightgbm(X: pl.DataFrame, y: np.ndarray, cat_features: list) -> list[str]:
    """Отбирает TOP_K_FEATURES наиболее важных признаков с помощью LightGBM (GPU)"""
    feature_names = X.columns
    X_pd = X.to_pandas()
    
    # Преобразуем категориальные признаки
    for col in cat_features:
        if col in X_pd.columns:
            X_pd[col] = X_pd[col].fillna(-1).astype(int)
    
    # Разделяем на train/val
    train_idx, valid_idx = train_test_split(
        np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y
    )
    
    X_train, X_val = X_pd.iloc[train_idx], X_pd.iloc[valid_idx]
    y_train, y_val = y[train_idx], y[valid_idx]
    
    # Категориальные индексы
    cat_indices = [X_train.columns.get_loc(col) for col in cat_features if col in X_train.columns]
    
    # Создаем datasets
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_indices)
    val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_indices, reference=train_data)
    
    # Параметры с GPU
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'verbose': -1,
        'num_threads': 8,
        'seed': SEED,
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0
    }
    
    # Обучаем
    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=100,
        callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
    )
    
    # Получаем важность признаков
    importance = model.feature_importance(importance_type='gain')
    importance_dict = dict(zip(feature_names, importance))
    
    # Сортируем и возвращаем топ признаков
    sorted_features = sorted(feature_names, key=lambda f: importance_dict.get(f, 0.0), reverse=True)
    return sorted_features[:TOP_K_FEATURES]

def save_selected_features(path: str, features: list[str]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(features))

def load_selected_features(target_name: str) -> list[str]:
    feature_path = f"{SELECTED_FEATURES_PATH}/{target_name}.txt"
    if os.path.exists(feature_path):
        with open(feature_path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f.readlines()]
    return None

def select_features_for_all_targets():
    """Выполняет отбор признаков для всех таргетов"""
    print("\n" + "="*60)
    print("ЭТАП 1: ОТБОР ПРИЗНАКОВ (LightGBM)")
    print("="*60)
    
    target = pl.read_parquet('data/train_target.parquet')
    target_cols = [col for col in target.columns if col.startswith("target")]
    cat_features = [col for col in train_wide.columns if col.startswith("cat_feature")]
    
    for target_name in target_cols:
        print(f"\n----- {target_name} -----")
        feature_path = f"{SELECTED_FEATURES_PATH}/{target_name}.txt"
        
        if os.path.exists(feature_path):
            print(f"✓ Уже существует, пропускаем")
            continue
        
        y = pl.read_parquet('data/train_target.parquet', columns=[target_name])[target_name].to_numpy()
        print(f"  Положительных: {int(y.sum())}, всего: {len(y)}")
        
        idx = subsample(SUBSAMPLE_SIZE, y)
        print(f"  Подвыборка: {len(idx)}")
        
        X = load_parquet_subsample('data/train_wide_features.parquet', idx)
        print(f"  Признаков: {X.shape[1]}")
        
        try:
            selected_features = select_topk_features_lightgbm(X, y[idx], cat_features)
            print(f"  Выбрано: {len(selected_features)}")
            save_selected_features(feature_path, selected_features)
            print(f"  ✓ Сохранено")
        except Exception as e:
            print(f"  ❌ Ошибка: {e}")
            save_selected_features(feature_path, [])

In [30]:
# ============================================================
# 3. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
# ============================================================
target = pl.read_parquet('data/train_target.parquet')
target_cols = [col for col in target.columns if col.startswith("target")]
y_all = target.select(target_cols).to_pandas()

cat_features = [col for col in train_wide.columns if col.startswith("cat_feature")]
print(f"\nКатегориальных признаков: {len(cat_features)}")

def fill_missing(df, cat_features):
    df = df.clone()
    num_cols = [col for col in df.columns if col not in cat_features]
    
    for col in num_cols:
        median_val = df[col].drop_nulls().median()
        if median_val is not None and not np.isnan(median_val):
            df = df.with_columns(pl.col(col).fill_null(median_val))
        else:
            df = df.with_columns(pl.col(col).fill_null(0))
    
    for col in cat_features:
        df = df.with_columns(pl.col(col).fill_null(-1).cast(pl.Int32))
    
    return df

print("\nЗаполнение пропусков...")
train_wide = fill_missing(train_wide, cat_features)
test_wide = fill_missing(test_wide, cat_features)

X = train_wide.to_pandas()
X_test = test_wide.to_pandas()


Категориальных признаков: 67

Заполнение пропусков...


In [31]:
# ------------------------------
# 5. Разбиение на группы
# ------------------------------
num_groups = 4
group_size = len(target_cols) // num_groups
groups = [target_cols[i*group_size:(i+1)*group_size] for i in range(num_groups)]
# Добавляем остаток в последнюю группу
if len(groups[-1]) < len(target_cols) - (num_groups-1)*group_size:
    groups[-1].extend(target_cols[num_groups*group_size:])
print("Группы целевых переменных:")
for i, g in enumerate(groups):
    print(f"  Группа {i+1}: {len(g)} классов")

Группы целевых переменных:
  Группа 1: 10 классов
  Группа 2: 10 классов
  Группа 3: 10 классов
  Группа 4: 11 классов


In [42]:
# Проверяем наличие отобранных признаков
need_selection = False
for target_name in target_cols[:5]:
    if not os.path.exists(f"{SELECTED_FEATURES_PATH}/{target_name}.txt"):
        need_selection = True
        break

if need_selection:
    print("\n⚠️ Запускаем отбор признаков...")
    select_features_for_all_targets()


⚠️ Запускаем отбор признаков...

ЭТАП 1: ОТБОР ПРИЗНАКОВ (LightGBM)

----- target_1_1 -----
  Положительных: 7797, всего: 750000
  Подвыборка: 120000
  Признаков: 2440
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[65]	valid_0's auc: 0.91165
  Выбрано: 300
  ✓ Сохранено

----- target_1_2 -----
  Положительных: 2569, всего: 750000
  Подвыборка: 120000
  Признаков: 2440
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[66]	valid_0's auc: 0.830802
  Выбрано: 300
  ✓ Сохранено

----- target_1_3 -----
  Положительных: 17839, всего: 750000
  Подвыборка: 120000
  Признаков: 2440
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[86]	valid_0's auc: 0.873848
  Выбрано: 300
  ✓ Сохранено

----- target_1_4 -----
  Положительных: 17572, всего: 750000
  Подвыборка: 120000
  Признаков: 2440
Training until validation scores don't improve for 20 r

In [44]:
# ============================================================
# 5. БЕЗОПАСНОЕ ОБУЧЕНИЕ С КРОСС-ВАЛИДАЦИЕЙ
# ============================================================
print("\n" + "="*60)
print("ОБУЧЕНИЕ LIGHTGBM")
print("="*60)

oof_predictions_all = {}
test_predictions_all = {}
oof_auc_scores = {}

def safe_train_fold(X_tr, y_tr, X_val, y_val, cat_indices, target_name):
    """Безопасное обучение с обработкой ошибок"""
    
    # Проверяем, что есть оба класса
    if len(np.unique(y_tr)) < 2:
        print(f"    ⚠️ В тренировке только один класс, возвращаем baseline")
        return None, np.mean(y_tr)
    
    # Проверяем размеры
    if len(y_tr) < 10 or len(y_val) < 5:
        print(f"    ⚠️ Слишком мало данных, пропускаем")
        return None, np.mean(y_tr)
    
    # Выбираем параметры
    pos_ratio = y_tr.mean()
    if pos_ratio < 0.05 or pos_ratio > 0.95:
        params = LGB_PARAMS_DEEP.copy()
    else:
        params = LGB_PARAMS.copy()
    
    # Добавляем веса
    pos = y_tr.sum()
    neg = len(y_tr) - pos
    if pos > 0 and neg > 0:
        params['scale_pos_weight'] = neg / pos
    
    # Уменьшаем min_data_in_leaf для редких классов
    if pos_ratio < 0.01:
        params['min_data_in_leaf'] = 5
        params['min_child_samples'] = 5
    
    try:
        train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_indices)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_indices, reference=train_data)
        
        model = lgb.train(
            params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=300,
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
        )
        
        pred_val = model.predict(X_val, num_iteration=model.best_iteration)
        return model, pred_val
        
    except Exception as e:
        print(f"    ❌ Ошибка: {e}")
        return None, np.ones(len(y_val)) * y_tr.mean()

for target_name in target_cols:
    print(f"\n{'='*50}")
    print(f"{target_name}")
    
    selected = load_selected_features(target_name)
    if not selected or len(selected) == 0:
        print(f"  ❌ Нет признаков, пропускаем")
        continue
    
    available = [f for f in selected if f in X.columns]
    if len(available) == 0:
        print(f"  ❌ Нет доступных признаков")
        continue
    
    cat_sel = [f for f in cat_features if f in available]
    
    X_data = X[available].copy()
    for col in cat_sel:
        X_data[col] = X_data[col].fillna(-1).astype(int)
    
    y_data = y_all[target_name].values
    
    # Пропускаем классы с очень редкими событиями
    if y_data.sum() < 10:
        print(f"  ⚠️ Слишком мало положительных ({y_data.sum()}), пропускаем")
        continue
    
    # Создаем фолды с проверкой
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    cat_indices = [X_data.columns.get_loc(col) for col in cat_sel if col in X_data.columns]
    
    oof_preds = np.zeros(len(X_data))
    test_preds = np.zeros(len(X_test))
    fold_scores = []
    success_folds = 0
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
        X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
        y_tr, y_val = y_data[train_idx], y_data[val_idx]
        
        # Проверяем наличие обоих классов
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_val)) < 2:
            print(f"  Fold {fold+1}: ⚠️ Пропущен (нет обоих классов)")
            continue
        
        model, pred_val = safe_train_fold(X_tr, y_tr, X_val, y_val, cat_indices, target_name)
        
        if model is not None and pred_val is not None:
            oof_preds[val_idx] = pred_val
            fold_score = roc_auc_score(y_val, pred_val)
            fold_scores.append(fold_score)
            success_folds += 1
            print(f"  Fold {fold+1}: AUC={fold_score:.4f}")
            
            # Тест
            X_test_sel = X_test[available].copy()
            for col in cat_sel:
                X_test_sel[col] = X_test_sel[col].fillna(-1).astype(int)
            test_preds += model.predict(X_test_sel, num_iteration=model.best_iteration) / N_FOLDS
    
    if success_folds > 0:
        oof_auc = roc_auc_score(y_data, oof_preds)
        oof_auc_scores[target_name] = oof_auc
        print(f"\n  ✅ OOF AUC: {oof_auc:.4f} (успешных фолдов: {success_folds}/{N_FOLDS})")
        
        oof_predictions_all[target_name] = oof_preds
        test_predictions_all[target_name] = test_preds
    else:
        print(f"  ❌ Не удалось обучить ни одного фолда")



ОБУЧЕНИЕ LIGHTGBM

target_1_1
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[104]	valid_0's auc: 0.904126
  Fold 1: AUC=0.9041
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[109]	valid_0's auc: 0.897081
  Fold 2: AUC=0.8971
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[114]	valid_0's auc: 0.895142
  Fold 3: AUC=0.8951

  ✅ OOF AUC: 0.8987 (успешных фолдов: 3/3)

target_1_2
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.747384
  Fold 1: AUC=0.7474
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.755232
  Fold 2: AUC=0.7552
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.756821
  Fold 3: AUC=0.7568

  ✅ OOF AUC: 0.7529 (успешных фолдов

In [46]:
# ============================================================
# 5. ФИНАЛЬНАЯ МОДЕЛЬ С МЕТА-ФИЧАМИ (для сложных классов) - ИСПРАВЛЕННАЯ
# ============================================================
print("\n" + "="*60)
print("ЭТАП 3: ФИНАЛЬНАЯ МОДЕЛЬ С МЕТА-ФИЧАМИ")
print("="*60)

# Определяем сложные классы (AUC < 0.75)
hard_classes = [name for name, auc in oof_auc_scores.items() if auc < 0.75]
print(f"Сложных классов: {len(hard_classes)}")

if hard_classes:
    print(f"Список: {hard_classes[:10]}...")
    
    # Считаем корреляции между таргетами
    target_corr = y_all[target_cols].corr()
    
    final_predictions = test_predictions_all.copy()
    
    for target_name in hard_classes:
        print(f"\nУлучшаем {target_name}...", end=" ", flush=True)
        
        selected_features = load_selected_features(target_name)
        if not selected_features:
            print("нет признаков")
            continue
        
        available_features = [f for f in selected_features if f in X.columns]
        if len(available_features) == 0:
            print("нет доступных признаков")
            continue
            
        cat_features_selected = [f for f in cat_features if f in available_features]
        
        # Находим топ-5 коррелирующих таргетов
        corr_others = target_corr[target_name].drop(target_name)
        top_targets = corr_others.nlargest(5).index.tolist()
        
        # Добавляем их OOF предсказания как фичи
        X_train_meta = X[available_features].copy()
        X_test_meta = X_test[available_features].copy()
        
        for other in top_targets:
            if other in oof_predictions_all and other in test_predictions_all:
                X_train_meta[f'meta_{other}'] = oof_predictions_all[other]
                X_test_meta[f'meta_{other}'] = test_predictions_all[other]
        
        # Преобразуем категориальные
        for col in cat_features_selected:
            if col in X_train_meta.columns:
                X_train_meta[col] = X_train_meta[col].fillna(-1).astype(int)
                X_test_meta[col] = X_test_meta[col].fillna(-1).astype(int)
        
        y_target = y_all[target_name].values
        
        # Проверяем наличие обоих классов
        if len(np.unique(y_target)) < 2:
            print("только один класс, пропускаем")
            continue
        
        # Проверяем минимальное количество положительных
        pos_count = y_target.sum()
        if pos_count < 10:
            print(f"мало положительных ({pos_count}), пропускаем")
            continue
        
        # Категориальные индексы
        cat_indices = [X_train_meta.columns.get_loc(col) for col in cat_features_selected if col in X_train_meta.columns]
        
        # Параметры для финальной модели (более консервативные)
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'num_leaves': 31,              # Уменьшаем
            'learning_rate': 0.03,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1,
            'num_threads': 8,
            'min_data_in_leaf': 10,        # Увеличиваем для стабильности
            'min_child_samples': 10,
            'lambda_l1': 0.1,
            'lambda_l2': 0.1,
            'seed': SEED
        }
        
        # Добавляем веса классов
        pos = y_target.sum()
        neg = len(y_target) - pos
        if pos > 0 and neg > 0:
            params['scale_pos_weight'] = neg / pos
        
        # Обучаем финальную модель с обработкой ошибок
        try:
            train_data = lgb.Dataset(X_train_meta, label=y_target, categorical_feature=cat_indices)
            
            model = lgb.train(
                params,
                train_data,
                num_boost_round=200,  # Уменьшаем
                callbacks=[lgb.log_evaluation(0)]
            )
            
            # Предсказания
            final_predictions[target_name] = model.predict(X_test_meta, num_iteration=model.best_iteration)
            print("готово ✅")
            
        except Exception as e:
            print(f"ошибка: {e}")
            # Оставляем исходные предсказания
            continue
    
    test_predictions_all = final_predictions


ЭТАП 3: ФИНАЛЬНАЯ МОДЕЛЬ С МЕТА-ФИЧАМИ
Сложных классов: 19
Список: ['target_2_3', 'target_2_4', 'target_2_5', 'target_2_6', 'target_2_7', 'target_3_1', 'target_3_3', 'target_3_4', 'target_5_1', 'target_5_2']...

Улучшаем target_2_3... готово ✅

Улучшаем target_2_4... готово ✅

Улучшаем target_2_5... готово ✅

Улучшаем target_2_6... готово ✅

Улучшаем target_2_7... готово ✅

Улучшаем target_3_1... готово ✅

Улучшаем target_3_3... готово ✅

Улучшаем target_3_4... готово ✅

Улучшаем target_5_1... готово ✅

Улучшаем target_5_2... готово ✅

Улучшаем target_6_1... готово ✅

Улучшаем target_6_2... готово ✅

Улучшаем target_6_3... готово ✅

Улучшаем target_6_5... готово ✅

Улучшаем target_7_3... готово ✅

Улучшаем target_9_1... готово ✅

Улучшаем target_9_3... готово ✅

Улучшаем target_9_6... готово ✅

Улучшаем target_10_1... готово ✅


In [47]:
# ============================================================
# 6. ФОРМИРОВАНИЕ ФИНАЛЬНОГО САБМИТА
# ============================================================
print("\n" + "="*60)
print("ЭТАП 4: ФОРМИРОВАНИЕ САБМИТА")
print("="*60)

# Собираем предсказания в правильном порядке
predictions = []
for target_name in target_cols:
    if target_name in test_predictions_all:
        predictions.append(test_predictions_all[target_name])
    else:
        # Fallback: предсказываем среднее значение (базовый уровень)
        mean_pred = y_all[target_name].mean()
        predictions.append(np.ones(len(X_test)) * mean_pred)

predictions = np.column_stack(predictions)
predict_cols = [f"predict_{col.replace('target_', '')}" for col in target_cols]
pred_df = pl.DataFrame(predictions, schema=predict_cols)

# Сохраняем
submit = pl.DataFrame({'customer_id': test_customer_ids.to_pandas()['customer_id']}).hstack(pred_df)
submit.write_parquet("data/sample_submit_lightgbm.parquet")

print("\n" + "="*60)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*60)

if oof_auc_scores:
    mean_oof = np.mean(list(oof_auc_scores.values()))
    print(f"Средний OOF AUC: {mean_oof:.4f}")
    
    # Статистика по классам
    sorted_scores = sorted(oof_auc_scores.items(), key=lambda x: x[1], reverse=True)
    print("\nТоп-5 лучших классов:")
    for name, auc in sorted_scores[:5]:
        print(f"  {name}: {auc:.4f}")
    
    print("\nТоп-5 худших классов:")
    for name, auc in sorted_scores[-5:]:
        print(f"  {name}: {auc:.4f}")

print(f"\n✅ Сабмит сохранён: data/sample_submit_lightgbm.parquet")
print(f"   Размер: {submit.shape[0]} строк, {submit.shape[1]} колонок")


ЭТАП 4: ФОРМИРОВАНИЕ САБМИТА

ИТОГОВЫЕ РЕЗУЛЬТАТЫ
Средний OOF AUC: 0.7582

Топ-5 лучших классов:
  target_8_1: 0.9752
  target_2_2: 0.9277
  target_9_8: 0.9229
  target_3_2: 0.9077
  target_1_1: 0.8987

Топ-5 худших классов:
  target_9_3: 0.6335
  target_6_1: 0.6302
  target_5_2: 0.6294
  target_2_7: 0.5920
  target_2_3: 0.5882

✅ Сабмит сохранён: data/sample_submit_lightgbm.parquet
   Размер: 250000 строк, 42 колонок
